# Notebook 06: Causal Inference from Observational Data

### Purpose
- In a randomized experiment, treatment and control groups are comparable by design. This notebook demonstrates what happens when they are not.
- Selection bias is artificially introduced into the Hillstrom dataset, making customers with high prior spend and recent purchase history more likely to receive the email.
- Causal inference methods are then applied to recover the true effect, using the experimental result from notebook 02 as ground truth.

### Objectives
- The degree to which selection bias inflates effect estimates when treatment assignment depends on customer characteristics.
- How well propensity score matching, inverse probability weighting, and doubly robust estimation correct for that bias.
- Which methods come closest to the true effect, and where residual bias remains.

In [20]:
import pandas as pd
import numpy as np

from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import ttest_ind
from scipy.special import expit  # sigmoid
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
import statsmodels.formula.api as smf

np.random.seed(42)

In [21]:
hillstrom_df = pd.read_csv("../data/raw/hillstrom.csv")

display(hillstrom_df.head(), hillstrom_df.tail())

display(hillstrom_df.info(), hillstrom_df.describe())

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
63995,10,2) $100 - $200,105.54,1,0,Urban,0,Web,Mens E-Mail,0,0,0.0
63996,5,1) $0 - $100,38.91,0,1,Urban,1,Phone,Mens E-Mail,0,0,0.0
63997,6,1) $0 - $100,29.99,1,0,Urban,1,Phone,Mens E-Mail,0,0,0.0
63998,1,5) $500 - $750,552.94,1,0,Surburban,1,Multichannel,Womens E-Mail,0,0,0.0
63999,1,4) $350 - $500,472.82,0,1,Surburban,0,Web,Mens E-Mail,0,0,0.0


<class 'pandas.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   recency          64000 non-null  int64  
 1   history_segment  64000 non-null  str    
 2   history          64000 non-null  float64
 3   mens             64000 non-null  int64  
 4   womens           64000 non-null  int64  
 5   zip_code         64000 non-null  str    
 6   newbie           64000 non-null  int64  
 7   channel          64000 non-null  str    
 8   segment          64000 non-null  str    
 9   visit            64000 non-null  int64  
 10  conversion       64000 non-null  int64  
 11  spend            64000 non-null  float64
dtypes: float64(2), int64(6), str(4)
memory usage: 5.9 MB


None

,recency,history,mens,womens,newbie,visit,conversion,spend
count,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000
mean,5.763734,242.085656,0.551031,0.549719,0.502250,0.146781,0.009031,1.050908
std,3.507592,256.158608,0.497393,0.497526,0.499999,0.353890,0.094604,15.036448
min,1.000000,29.990000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,64.660000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,6.000000,158.110000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
75%,9.000000,325.657500,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
max,12.000000,3345.930000,1.000000,1.000000,1.000000,1.000000,1.000000,499.000000


## Experimental ATE (also calculated in notebook 02)

In [22]:
mens_control_df = hillstrom_df[hillstrom_df["segment"].isin(['Mens E-Mail', 'No E-Mail'])]

In [23]:
# Initialize an empty DataFrame to store results
results_df = pd.DataFrame(columns=["method", "visit", "conversion", "spend"])

In [24]:
# z test for proportions

results = []

for metric in ["visit", "conversion"]:

    ate = mens_control_df[mens_control_df["segment"] == "Mens E-Mail"][metric].mean() - mens_control_df[mens_control_df["segment"] == "No E-Mail"][metric].mean()

    counts_df = mens_control_df.groupby("segment")[metric].agg(["sum", "count"])

    zstat, pvalue = proportions_ztest([counts_df.loc["Mens E-Mail", "sum"], counts_df.loc["No E-Mail", "sum"]],
                                       [counts_df.loc["Mens E-Mail", "count"], counts_df.loc["No E-Mail", "count"]])

    p1 = counts_df.loc["Mens E-Mail", "sum"] / counts_df.loc["Mens E-Mail", "count"]
    p2 = counts_df.loc["No E-Mail", "sum"] / counts_df.loc["No E-Mail", "count"]
    n1 = counts_df.loc["Mens E-Mail", "count"]
    n2 = counts_df.loc["No E-Mail", "count"]
    var_dat_1 = p1 * (1 - p1)
    var_dat_2 = p2 * (1 - p2)
    se_diff = np.sqrt(var_dat_1 / n1 + var_dat_2 / n2)
    ci = (p1 - p2) + np.array([-1, 1]) * 1.96 * se_diff
    ci = ci.round(4)

    results.append({
        "Metric": metric, "ATE": ate, "95% CI": ci, "Z-Statistic": zstat, "P-Value": pvalue, 
    })

exp_ates = {r["Metric"]: r["ATE"] for r in results}

pd.DataFrame(results)

,Metric,ATE,95% CI,Z-Statistic,P-Value
0,visit,0.076590,"[0.07, 0.0832]",22.486041,5.685165e-112
1,conversion,0.006805,"[0.005, 0.0086]",7.385114,1.523224e-13


In [25]:
# Spend

metric = "spend"

spend1 = mens_control_df[mens_control_df["segment"] == "Mens E-Mail"][metric]
spend2 = mens_control_df[mens_control_df["segment"] == "No E-Mail"][metric]

tstat, pvalue = ttest_ind(spend1, spend2, equal_var=False)

ate = spend1.mean() - spend2.mean()
var_dat_1 = np.sum((spend1 - spend1.mean()) ** 2) / (len(spend1) - 1)
var_dat_2 = np.sum((spend2 - spend2.mean()) ** 2) / (len(spend2) - 1)
var_mean_1 = var_dat_1 / len(spend1)
var_mean_2 = var_dat_2 / len(spend2)
se_diff = np.sqrt(var_mean_1 + var_mean_2)
ci = ate + np.array([-1, 1]) * 1.96 * se_diff
ci = ci.round(4)

exp_ates["spend"] = ate
results_df = pd.concat([results_df, pd.DataFrame([{"method": "Experimental (ground truth)", **exp_ates}])], ignore_index=True)

results = [{
        "Metric": metric, "ATE": ate, "95% CI": ci, "T-Statistic": tstat, "P-Value": pvalue,
    }]

pd.DataFrame(results)

,Metric,ATE,95% CI,T-Statistic,P-Value
0,spend,0.769827,"[0.4851, 1.0545]",5.30014,1.163815e-07


## Introduce artificial selection bias
- **Bias mechanism:** more recent customers (low recency) and higher prior spend are more likely to be in the treatment group - simulating self-selection into a marketing email list

In [27]:
mens_control_df = hillstrom_df.loc[hillstrom_df["segment"].isin(["Mens E-Mail", "No E-Mail"])].copy()
mens_control_df["treatment"] = (mens_control_df["segment"] == "Mens E-Mail").astype(int)

# Standardize covariates
recency_std = (mens_control_df["recency"] - mens_control_df["recency"].mean()) / mens_control_df["recency"].std()
history_std = (mens_control_df["history"] - mens_control_df["history"].mean()) / mens_control_df["history"].std()

# Bias mechanism: more recent customers (low recency) and higher prior spend
# are more likely to be in the treatment group — simulating self-selection
# into a marketing email list
selection_score = -0.5 * recency_std + 0.5 * history_std
selection_prob = expit(selection_score)

# Keep all control, keep treated with probability = selection_prob
is_control = mens_control_df["treatment"] == 0
is_kept_treated = (mens_control_df["treatment"] == 1) & (np.random.uniform(size=len(mens_control_df)) < selection_prob)

biased_df = mens_control_df[is_control | is_kept_treated].copy()

display(biased_df.head(), biased_df.tail())


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend,treatment
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0,0
14,4,3) $200 - $350,241.42,0,1,Rural,1,Multichannel,No E-Mail,0,0,0.0,0
15,3,1) $0 - $100,58.13,1,0,Urban,1,Web,No E-Mail,1,0,0.0,0
16,5,1) $0 - $100,29.99,1,0,Surburban,0,Phone,Mens E-Mail,0,0,0.0,1
19,5,"6) $750 - $1,000",828.42,1,0,Surburban,1,Multichannel,Mens E-Mail,0,0,0.0,1


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend,treatment
63983,2,1) $0 - $100,83.03,0,1,Urban,0,Phone,No E-Mail,0,0,0.0,0
63987,1,1) $0 - $100,79.70,1,0,Surburban,1,Web,No E-Mail,0,0,0.0,0
63988,6,1) $0 - $100,32.98,1,0,Surburban,0,Web,Mens E-Mail,0,0,0.0,1
63990,6,1) $0 - $100,80.02,0,1,Surburban,0,Phone,No E-Mail,0,0,0.0,0
63992,1,5) $500 - $750,519.69,1,1,Urban,1,Phone,Mens E-Mail,0,0,0.0,1


### Bias check
- Check the mean recency and history between treatment and control in both the original and biased datasets. In the original they should be balanced (close to equal), in the biased dataset the treated group should have lower recency and higher history than control.

In [28]:
for df, label in [(mens_control_df, "Original"), (biased_df, "Biased")]:
    print(label)
    display(df.groupby("treatment")[["recency", "history"]].mean())

Original


,recency,history
treatment,,
0,5.749695,240.882653
1,5.773642,242.835931


Biased


,recency,history
treatment,,
0,5.749695,240.882653
1,4.772452,307.859734


## ATE after artificial selection bias

In [29]:
# z test for proportions

results = []

for metric in ["visit", "conversion"]:

    ate = biased_df[biased_df["segment"] == "Mens E-Mail"][metric].mean() - biased_df[biased_df["segment"] == "No E-Mail"][metric].mean()

    counts_df = biased_df.groupby("segment")[metric].agg(["sum", "count"])

    zstat, pvalue = proportions_ztest([counts_df.loc["Mens E-Mail", "sum"], counts_df.loc["No E-Mail", "sum"]],
                                       [counts_df.loc["Mens E-Mail", "count"], counts_df.loc["No E-Mail", "count"]])

    p1 = counts_df.loc["Mens E-Mail", "sum"] / counts_df.loc["Mens E-Mail", "count"]
    p2 = counts_df.loc["No E-Mail", "sum"] / counts_df.loc["No E-Mail", "count"]
    n1 = counts_df.loc["Mens E-Mail", "count"]
    n2 = counts_df.loc["No E-Mail", "count"]
    var_dat_1 = p1 * (1 - p1)
    var_dat_2 = p2 * (1 - p2)
    se_diff = np.sqrt(var_dat_1 / n1 + var_dat_2 / n2)
    ci = (p1 - p2) + np.array([-1, 1]) * 1.96 * se_diff
    ci = ci.round(4)

    results.append({
        "Metric": metric, "ATE": ate, "95% CI": ci, "Z-Statistic": zstat, "P-Value": pvalue,
    })

naive_ates = {r["Metric"]: r["ATE"] for r in results}

pd.DataFrame(results)

,Metric,ATE,95% CI,Z-Statistic,P-Value
0,visit,0.087372,"[0.0788, 0.096]",21.473687,2.743654e-102
1,conversion,0.008863,"[0.0064, 0.0114]",8.035352,9.331027e-16


In [30]:
# Spend

metric = "spend"

spend1 = biased_df[biased_df["segment"] == "Mens E-Mail"][metric]
spend2 = biased_df[biased_df["segment"] == "No E-Mail"][metric]

tstat, pvalue = ttest_ind(spend1, spend2, equal_var=False)

ate = spend1.mean() - spend2.mean()
var_dat_1 = np.sum((spend1 - spend1.mean()) ** 2) / (len(spend1) - 1)
var_dat_2 = np.sum((spend2 - spend2.mean()) ** 2) / (len(spend2) - 1)
var_mean_1 = var_dat_1 / len(spend1)
var_mean_2 = var_dat_2 / len(spend2)
se_diff = np.sqrt(var_mean_1 + var_mean_2)
ci = ate + np.array([-1, 1]) * 1.96 * se_diff
ci = ci.round(4)

naive_ates["spend"] = ate
results_df = pd.concat([results_df, pd.DataFrame([{"method": "Naive biased", **naive_ates}])], ignore_index=True)

results = [{
        "Metric": metric, "ATE": ate, "95% CI": ci, "T-Statistic": tstat, "P-Value": pvalue,
    }]

pd.DataFrame(results)

,Metric,ATE,95% CI,T-Statistic,P-Value
0,spend,0.98295,"[0.5887, 1.3772]",4.887287,0.000001


## Propensity score matching

In [31]:
# Step 1: Estimate propensity scores
X = biased_df[["recency", "history"]]
y = biased_df["treatment"]

lr = LogisticRegression()
lr.fit(X, y)
biased_df["propensity_score"] = lr.predict_proba(X)[:, 1]

# Step 2: Nearest neighbor matching — for each treated unit find closest control
treated = biased_df[biased_df["treatment"] == 1].copy()
control = biased_df[biased_df["treatment"] == 0].copy()

nn = NearestNeighbors(n_neighbors=1)
nn.fit(control[["propensity_score"]])
distances, indices = nn.kneighbors(treated[["propensity_score"]])

matched_control = control.iloc[indices.flatten()].copy()
matched_df = pd.concat([treated, matched_control])

# Step 3: Compute PSM ATEs and append to results
psm_ates = {"method": "Propensity score matching"}
results = []
for metric in ["visit", "conversion", "spend"]:
    ate = matched_df[matched_df["treatment"] == 1][metric].mean() - matched_df[matched_df["treatment"] == 0][metric].mean()
    psm_ates[metric] = ate
    results.append({"metric": metric, "psm_ate": ate})

results_df = pd.concat([results_df, pd.DataFrame([psm_ates])], ignore_index=True)

pd.DataFrame(results)

,metric,psm_ate
0,visit,0.076639
1,conversion,0.008336
2,spend,0.888703


## Inverse Probability Weighting

In [32]:
biased_df["ipw"] = np.where(
    biased_df["treatment"] == 1,
    1 / biased_df["propensity_score"],
    1 / (1 - biased_df["propensity_score"])
)

ipw_ates = {"method": "IPW"}
results = []
for metric in ["visit", "conversion", "spend"]:
    treated = biased_df[biased_df["treatment"] == 1]
    control = biased_df[biased_df["treatment"] == 0]
    ate = np.average(treated[metric], weights=treated["ipw"]) - np.average(control[metric], weights=control["ipw"])
    ipw_ates[metric] = ate
    results.append({"metric": metric, "ipw_ate": ate})

results_df = pd.concat([results_df, pd.DataFrame([ipw_ates])], ignore_index=True)

pd.DataFrame(results)

,metric,ipw_ate
0,visit,0.075859
1,conversion,0.007669
2,spend,0.818253


In [33]:
biased_df["propensity_score"].describe()

count    31862.000000
mean         0.331304
std          0.074244
min          0.208338
25%          0.271026
50%          0.329369
75%          0.375920
max          0.818159
Name: propensity_score, dtype: float64

### IPW diagnostics
- Range 0.21–0.82, no extreme values - positivity assumption satisfied, no weight trimming needed
- Narrow std (0.07) - recency and history are weak predictors of treatment, scores tightly clustered
- Consistent with weak selection bias - matching and IPW won't produce dramatic corrections

## Doubly robust estimation

In [34]:
dr_ates = {"method": "Doubly robust"}
results = []
for metric in ["visit", "conversion", "spend"]:
    model = smf.ols(f"{metric} ~ treatment + recency + history", data=biased_df).fit()

    df_treated = biased_df.copy()
    df_treated["treatment"] = 1
    df_control = biased_df.copy()
    df_control["treatment"] = 0

    mu_1 = model.predict(df_treated)
    mu_0 = model.predict(df_control)

    e = biased_df["propensity_score"]
    y = biased_df[metric]
    t = biased_df["treatment"]

    ate = (mu_1 - mu_0 + t * (y - mu_1) / e - (1 - t) * (y - mu_0) / (1 - e)).mean()
    dr_ates[metric] = ate
    results.append({"metric": metric, "dr_ate": ate})

results_df = pd.concat([results_df, pd.DataFrame([dr_ates])], ignore_index=True)

pd.DataFrame(results)

,metric,dr_ate
0,visit,0.075685
1,conversion,0.007652
2,spend,0.816332


## Comparison of the ATEs for the different methods

In [35]:
results_df.set_index("method").round(4)

,visit,conversion,spend
method,,,
Experimental (ground truth),0.07659,0.006805,0.769827
Naive biased,0.087372,0.008863,0.98295
Propensity score matching,0.076639,0.008336,0.888703
IPW,0.075859,0.007669,0.818253
Doubly robust,0.075685,0.007652,0.816332


In [37]:
gt = results_df[results_df["method"] == "Experimental (ground truth)"][["visit", "conversion", "spend"]].astype(float).values[0]

pct_df = results_df[results_df["method"] != "Experimental (ground truth)"].copy()
pct_df[["visit", "conversion", "spend"]] = pct_df[["visit", "conversion", "spend"]].astype(float)
pct_df[["visit", "conversion", "spend"]] = ((pct_df[["visit", "conversion", "spend"]].values - gt) / gt * 100).round(1)
pct_df = pct_df.set_index("method")
pct_df.columns = ["Visit % vs GT", "Conversion % vs GT", "Spend % vs GT"]
pct_df

,Visit % vs GT,Conversion % vs GT,Spend % vs GT
method,,,
Naive biased,14.1,30.2,27.7
Propensity score matching,0.1,22.5,15.4
IPW,-1.0,12.7,6.3
Doubly robust,-1.2,12.4,6.0


## Summary

- Introducing selection bias inflated the naive ATE across all outcomes: visit by 14%, conversion by 30%, and spend by 28%, driven by more engaged, higher-spending customers being over-represented in the treated group.
- Propensity score matching recovered the visit effect almost exactly (+0.1%) but remained substantially biased for conversion (+22%) and spend (+15%).
- IPW and doubly robust estimation performed similarly and best overall, reducing spend bias to ~6% and conversion bias to ~12% above ground truth.
- Conversion was the hardest metric to recover. Even with doubly robust estimation, 12% bias remained, consistent with weak propensity scores (mean 0.33, std 0.07) that lacked predictive power to fully account for the confounding.